# Information Extraction

#### Developed By: Manaranjan Pradhan
#### www.manaranjanp.com

*This Jupyter notebook is confidential and proprietary to Manaranjan Pradhan. It is intended solely for authorized training purposes. Unauthorized distribution, sharing, or reproduction of this notebook or its contents is strictly prohibited. This material is for personal learning within the training program only and may not be used for commercial purposes or shared with others. Unauthorized use may result in disciplinary action or legal consequences. If you have received this notebook without authorization, please contact manaranjan@gmail.com immediately and delete all copies.*



In [2]:
!pip install -q langchain langchain_groq PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.6 MB/s eta 0:00:00


## Accesing the Page Content

In [5]:
from PyPDF2 import PdfReader

def pdf_to_txt_pypdf2_string(pdf_path):
    """
    Reads a PDF file and returns its text content as a single string.
    Each page's content is separated by a newline.
    """
    full_text = []
    try:
        reader = PdfReader(pdf_path)
        for page in reader.pages:
            text = page.extract_text()
            if text:
                full_text.append(text)
    except Exception as e:
        print(f"Error processing PDF with PyPDF2: {e}")
        return None
    return "\n".join(full_text)

In [6]:
contract_txt = pdf_to_txt_pypdf2_string("/content/LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_Affiliate Agreement.pdf")

In [8]:
print(contract_txt[0:1000])

                                  EXHIBIT 10.1
                     AFFLIATE AGREEMENT DATED JULY 15, 2005
                               AFFILIATE AGREEMENT
         This Agreement entered into as of the Effective Date by and between
Link Plus Corporation  and Axiometric, LLC .
                                    RECITALS
         WHEREAS, Axiometric has developed certain computer software including
wireless mesh networking technology and AMR devices and systems;
         WHEREAS, LKPL has developed certain radio devices and systems along
with hardware manufacturing capacities and plans to develop AMR devices and
systems;
         WHEREAS, LKPL and Axiometric believe it will be in their mutual best
interests to cooperate in further developing AMR product suites by creating a
preferred provider relationship between themselves;
         WHEREAS, LKPL and Axiometric entered into a Letter of Intent dated May
3, 2005, and now desire to further describe their relationship as initially set
f

## Configuring LLM for Query

In [9]:
import os
from getpass import getpass
from pprint import pprint

In [10]:
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


### Initializing the OpenAI Chat Model

In [11]:
from langchain.chains import create_extraction_chain, create_extraction_chain_pydantic
from langchain_groq import ChatGroq

In [12]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=2048,
)

### Configuring the Retrieval QA Chain

https://json-schema.org/understanding-json-schema/

In [15]:
schema = {
  'type': 'function',
  'function': {
    'name': 'lease_contract_extractions',
    'description': '',
    'parameters': {
                "end_date": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "The end date of the lease.",
                },
                "leased_space": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Description of the space that is being leased.",
                },
                "lessee": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "The lessee's name (and possibly address).",
                },
                "lessor": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "The lessor's name (and possibly address).",
                },
                "signing_date": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "The date the contract was signed.",
                },
                "start_date": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "The start date of the lease.",
                },
                "term_of_payment": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Description of the payment terms.",
                },
                "designated_use": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Designated use of the property being leased.",
                },
                "extension_period": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Description of the extension options for the lease.",
                },
                "expiration_date_of_lease": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "The expiration date of the lease.",
                },
            },
            "required": [
                "end_date",
                "leased_space",
                "lessee",
                "lessor",
                "signing_date",
                "start_date",
                "term_of_payment",
                "designated_use",
                "extension_period",
                "expiration_date_of_lease",
            ],
        },
        "strict": True,
}

### Configurating Conversational Memory

In [16]:
#chain = create_extraction_chain(schema, llm)
structured_llm = llm.with_structured_output(schema)

## Asking Questions

In [17]:
response = structured_llm.invoke(contract_txt)

In [19]:
response

{'designated_use': ['office space'],
 'end_date': ['perpetuity'],
 'expiration_date_of_lease': ['until otherwise mutually agreed or amended in writing by both parties'],
 'extension_period': ['none'],
 'leased_space': ["office space in LKPL's corporate facility in Columbia, Maryland"],
 'lessee': ['Axiometric, LLC'],
 'lessor': ['Link Plus Corporation'],
 'signing_date': ['July 15, 2005'],
 'start_date': ['July 15, 2005'],
 'term_of_payment': ['45 days of the close of each calendar quarter']}